# PIPELINE LENGKAP: ANOTASI GEMINI + TRAINING INDOBERT + INFERENSI + SNA (KEMEN PU)
============================================================================
ALL-IN-ONE Google Colab Notebook dengan GPU T4
Proyek: Analisis Sentimen & SNA Opini Publik Kementerian PU & Menteri PU (Ir. Dody Hanggodo, M.P.E.)

CARA PAKAI:
  1. Buka Google Drive (akun yang sama dengan Google Colab)
  2. Buat folder baru bernama: IndoBERT_SNA_KEMENPU  (di MyDrive / Drive Saya)
  3. Upload file dari folder project lokal (IndoBERT_SNA_KEMENPU_MT_KEMENPU/data/):
     - master_scraped_corpus.csv      (Wajib - 15.836 data bersih 7 platform)
     - gold_standard_sample_1000.csv  (Wajib untuk Cell Anotasi Gemini - 1.000 sampel seimbang)
     - train.csv, val.csv, test.csv   (Opsional - dataset latih sentimen)
     - train_emotion_9.csv, val_emotion_9.csv, test_emotion_9.csv (Opsional - dataset latih 9 emosi)
  4. Buka notebook ini di Google Colab:
     - File -> Upload notebook -> pilih file IndoBERT_SNA_KEMENPU_KEMENPU_Training_Pipeline.ipynb
  5. Pastikan Hardware Acceleration aktif:
     - Klik menu Runtime -> Change runtime type -> Hardware accelerator: GPU (pilih T4 GPU) -> Save
  6. Jalankan Semua Cell:
     - Klik Runtime -> Run All (atau tekan tombol Ctrl + F9)
============================================================================


## CELL 1: INSTALL & SETUP


In [ ]:
!pip install transformers torch accelerate scikit-learn seaborn pandas pyarrow google-genai -q

import os, sys, time, json, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# pyrefly: ignore [missing-import]
# pyright: reportMissingImports=false
import torch  # type: ignore
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report
)
from sklearn.model_selection import train_test_split
from google import genai

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


## CELL 2: MOUNT DRIVE & CONFIG


In [ ]:
# pyrefly: ignore [missing-import]
# pyright: reportMissingImports=false
from google.colab import drive  # type: ignore
drive.mount('/content/drive')

import os, time, json, re
import pandas as pd
from google import genai

# Deteksi otomatis folder IndoBERT_SNA_KEMENPU di Google Drive
possible_dirs = [
    '/content/drive/MyDrive/IndoBERT_SNA_KEMENPU',
    '/content/drive/MyDrive/IndoBERT_SNA'
]
DRIVE_DIR = '/content/drive/MyDrive/IndoBERT_SNA_KEMENPU'
for d in possible_dirs:
    if os.path.exists(d):
        DRIVE_DIR = d
        break

os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"📁 Menggunakan direktori Google Drive: {DRIVE_DIR}")

# MASUKKAN 3 ATAU 4 API KEY ANDA DI SINI DALAM TANDA PETIK:
GEMINI_API_KEYS = [
    "YOUR_GEMINI_API_KEY_HERE",
    "YOUR_GEMINI_API_KEY_HERE",
    "YOUR_GEMINI_API_KEY_HERE",
    "YOUR_GEMINI_API_KEY_HERE",
    # 4 API Key Cadangan (Commented):
    # "YOUR_GEMINI_API_KEY_HERE",
    # "YOUR_GEMINI_API_KEY_HERE",
    # "YOUR_GEMINI_API_KEY_HERE",
    # "YOUR_GEMINI_API_KEY_HERE",
]

GOLD_SAMPLE_PATH = os.path.join(DRIVE_DIR, 'gold_standard_sample_1000.csv')
GOLD_ANNOTATED_PATH = os.path.join(DRIVE_DIR, 'gold_standard_annotated_1000.csv')
MASTER_PATH = os.path.join(DRIVE_DIR, 'master_scraped_corpus.csv')
ENRICHED_PATH = os.path.join(DRIVE_DIR, 'enriched_corpus_for_sna.parquet')
CHECKPOINT_PATH = os.path.join(DRIVE_DIR, 'annotation_checkpoint.json')

SENT_DIR = os.path.join(DRIVE_DIR, 'best_indobert_sentiment_v2')
EMO_DIR = os.path.join(DRIVE_DIR, 'best_indobert_emotion')

MODEL_NAME = "indobenchmark/indobert-base-p1"
SENTIMENT_LABELS = ['Negatif', 'Netral', 'Positif']

EMOTION_9_CLASSES = [
    'Trust', 'Anticipation', 'Joy',
    'Disgust', 'Fear', 'Sadness', 'Anger',
    'Surprise', 'Neutral'
]
EMOTION_LABELS = EMOTION_9_CLASSES
EMOTION_TO_SENTIMENT = {
    'Trust': 'Positif', 'Anticipation': 'Positif', 'Joy': 'Positif',
    'Disgust': 'Negatif', 'Fear': 'Negatif', 'Sadness': 'Negatif', 'Anger': 'Negatif',
    'Surprise': 'Netral', 'Neutral': 'Netral'
}


## CELL 3: GEMINI ANNOTATION (Langkah 1 & 2)


In [ ]:
# =====================================================================
# CELL 3: GEMINI ANNOTATION DENGAN MULTI-KEY ROTATION (FIX KEYERROR)
# =====================================================================
print("=" * 70)
print("LANGKAH 1: ANOTASI GOLD STANDARD DENGAN GEMINI")
print("=" * 70)

need_annotation = True
gold_df = pd.DataFrame()

if need_annotation:
    # 1. Siapkan pool client dari daftar API Keys
    clients = [genai.Client(api_key=k.strip()) for k in GEMINI_API_KEYS if k.strip() and not k.strip().startswith("KEY_")]
    print(f"🔑 Menggunakan pool {len(clients)} API Key bergantian (Round-Robin).")

    # 2. Cek model yang aktif
    working_model = 'gemini-3.7-flash'
    for m in ['gemini-2.0-flash', 'gemini-1.5-flash', 'gemini-2.5-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'gemini-3.1-flash-lite']:
        try:
            clients[0].models.generate_content(model=m, contents='1+1=?')
            working_model = m
            print(f"🤖 Model aktif: {m}")
            break
        except:
            continue

    PROMPT = """Kamu ahli analisis sentimen Bahasa Indonesia untuk kebijakan publik (Kementerian Pekerjaan Umum RI / Kemen PU dan Menteri PU).
Label setiap teks:
- sentiment: "Positif" / "Netral" / "Negatif"
- emotion: "Trust" / "Anticipation" / "Joy" / "Disgust" / "Fear" / "Sadness" / "Anger" / "Surprise" / "Neutral"
- confidence: 0.0-1.0
- reason: alasan max 5 kata

Output HANYA JSON array murni:
[{"idx":0,"sentiment":"...","emotion":"...","confidence":0.9,"reason":"..."}]

TEKS:
{texts}"""

    # 3. Load sampel 1.000 data
    if not os.path.exists(GOLD_SAMPLE_PATH):
        sample_src = pd.read_parquet(ENRICHED_PATH) if os.path.exists(ENRICHED_PATH) else pd.read_csv(MASTER_PATH, low_memory=False)
        df_sample = sample_src.sample(min(1000, len(sample_src)), random_state=42)
    else:
        df_sample = pd.read_csv(GOLD_SAMPLE_PATH)

    BATCH_SIZE = 10
    batches = [df_sample.iloc[i:i+BATCH_SIZE] for i in range(0, len(df_sample), BATCH_SIZE)]
    total_batches = len(batches)
    print(f"📋 Total data yang akan dianotasi: {len(df_sample):,} entri ({total_batches} batch)")

    # 4. Resume checkpoint jika ada progres tersimpan
    done_set, all_anns = set(), []
    if os.path.exists(CHECKPOINT_PATH):
        try:
            with open(CHECKPOINT_PATH, 'r') as f:
                cp = json.load(f)
            done_set = set(cp.get('done', []))
            all_anns = cp.get('ann', [])
            print(f"📂 Melanjutkan dari checkpoint: {len(all_anns)}/1,000 data sudah teranotasi ({len(done_set)} batch).")
        except:
            pass

    t0 = time.time()
    client_idx = 0
    print("\n🚀 Memulai proses anotasi...")

    # 5. Loop Anotasi Per 10 Data
    for bn, bdf in enumerate(batches):
        if bn in done_set:
            continue

        texts = "".join([f'\n[{i}] ({r.get("platform","web")}/{r.get("topic","kebijakan")}) "{str(r["cleaned_content"])[:300].replace(chr(34), chr(39))}"\n' for i, (_, r) in enumerate(bdf.iterrows())])
        
        # FIX: Gunakan .replace() agar kurung kurawal JSON tidak memicu KeyError
        prompt = PROMPT.replace("{texts}", texts)

        # Identifikasi API Key ke berapa yang sedang aktif
        active_key_num = (client_idx % len(clients)) + 1
        active_client = clients[client_idx % len(clients)]
        results = None

        tb0 = time.time()
        try:
            r = active_client.models.generate_content(
                model=working_model, 
                contents=prompt,
                config={
                    'temperature': 0.1,
                    'response_mime_type': 'application/json'  # Menjamin JSON selalu valid tanpa terpotong
                }
            )
            results = json.loads(r.text.strip())
        except Exception as e:
            err_msg = str(e)
            next_key_num = ((client_idx + 1) % len(clients)) + 1
            print(f"  ⚠️ Key-{active_key_num} kendala ({err_msg[:45]}...). Langsung coba ulang dengan Key-{next_key_num}!", flush=True)
            client_idx += 1
            active_key_num = next_key_num
            active_client = clients[client_idx % len(clients)]
            try:
                r = active_client.models.generate_content(
                    model=working_model, 
                    contents=prompt,
                    config={
                        'temperature': 0.1,
                        'response_mime_type': 'application/json'
                    }
                )
                results = json.loads(r.text.strip())
            except Exception as e2:
                try:
                    m = re.search(r'\[.*\]', r.text.strip(), re.DOTALL)
                    if m:
                        results = json.loads(m.group())
                except:
                    pass

        # Geser giliran key untuk batch berikutnya (Round-Robin)
        client_idx += 1

        if not isinstance(results, list): 
            results = []
        while len(results) < len(bdf):
            results.append({"sentiment": "Netral", "emotion": "Neutral", "confidence": 0.5, "reason": "auto-filled"})

        emo_norm_map = {
            'Percaya': 'Trust', 'Trust': 'Trust',
            'Antisipasi': 'Anticipation', 'Anticipation': 'Anticipation',
            'Senang': 'Joy', 'Joy': 'Joy',
            'Jijik': 'Disgust', 'Disgust': 'Disgust',
            'Takut': 'Fear', 'Fear': 'Fear',
            'Sedih': 'Sadness', 'Sadness': 'Sadness',
            'Marah': 'Anger', 'Anger': 'Anger',
            'Terkejut': 'Surprise', 'Surprise': 'Surprise',
            'Informatif': 'Neutral', 'Netral': 'Neutral', 'Neutral': 'Neutral'
        }
        for i, (di, row) in enumerate(bdf.iterrows()):
            res = results[i] if i < len(results) and isinstance(results[i], dict) else {}
            s_val = res.get('sentiment', 'Netral') if res.get('sentiment') in {'Positif', 'Negatif', 'Netral'} else 'Netral'
            e_raw = res.get('emotion', 'Neutral')
            e_val = emo_norm_map.get(e_raw, 'Neutral') if emo_norm_map.get(e_raw, 'Neutral') in EMOTION_9_CLASSES else 'Neutral'
            all_anns.append({
                "original_index": int(di), "platform": row.get('platform', ''), "topic": row.get('topic', ''),
                "cleaned_content": str(row['cleaned_content'])[:500], "old_sentiment": str(row.get('sentiment_name', '')),
                "old_emotion": str(row.get('emotion', '')), "gemini_sentiment": s_val, "gemini_emotion": e_val,
                "gemini_confidence": float(res.get('confidence', 0.8)), "gemini_reason": str(res.get('reason', ''))
            })
        done_set.add(bn)

        # Simpan progres berkala ke Google Drive
        with open(CHECKPOINT_PATH, 'w') as f:
            json.dump({"done": list(done_set), "ann": all_anns}, f, ensure_ascii=False)

        # Report log per 10 data
        pct = len(done_set) / total_batches * 100
        batch_sec = time.time() - tb0
        print(f"  Batch {bn+1:02d}/{total_batches} ({pct:3.0f}%) | Key-{active_key_num} | {batch_sec:4.1f}s | Total: {len(all_anns):,}/1,000 data dianotasi", flush=True)

        time.sleep(1.0)

    # 6. Simpan File Final
    if all_anns:
        gold_df = pd.DataFrame(all_anns)
        gold_df['sentiment_label'] = gold_df['gemini_sentiment'].map({'Negatif': 0, 'Netral': 1, 'Positif': 2})
        gold_df['emotion_label'] = gold_df['gemini_emotion'].map({c: i for i, c in enumerate(EMOTION_9_CLASSES)})
        gold_df.to_csv(GOLD_ANNOTATED_PATH, index=False, encoding='utf-8-sig')
        print(f"\n{'='*70}")
        print(f"✅ Anotasi 1.000 data SELESAI ({time.time()-t0:.0f}s)! Disimpan ke: {GOLD_ANNOTATED_PATH}")
        print(f"{'='*70}")
        print("Distribusi Sentimen:\n" + gold_df['gemini_sentiment'].value_counts().to_string())
        print("\nDistribusi Emosi:\n" + gold_df['gemini_emotion'].value_counts().to_string())


## CELL 4: CROSS-VALIDATE (Langkah 2)


In [ ]:
print("\n" + "=" * 70)
print("LANGKAH 2: CROSS-VALIDATE OLD vs GEMINI")
print("=" * 70)

if not gold_df.empty and 'gemini_sentiment' in gold_df.columns and len(gold_df) >= 10:
    ct = pd.crosstab(gold_df['old_sentiment'], gold_df['gemini_sentiment'])
    print("Cross-tabulation (Lama vs Gemini):")
    print(ct.to_string())
    old_acc = (gold_df['old_sentiment'] == gold_df['gemini_sentiment']).mean() * 100
    print(f"\nAkurasi rule-based lama vs Gemini: {old_acc:.1f}%")
else:
    print("Anotasi Gemini terbatas. Mengecek distribusi sentimen korpus:")
    if os.path.exists(ENRICHED_PATH):
        chk = pd.read_parquet(ENRICHED_PATH)
        print(chk['predicted_sentiment'].value_counts().to_string())


## CELL 5: PREPARE & TRAIN SENTIMENT MODEL (Langkah 3)


In [ ]:
print("\n" + "=" * 70)
print("LANGKAH 3: TRAINING INDOBERT V2 - SENTIMENT")
print("=" * 70)

# Mode Seimbang Anti-Bias: Selalu gunakan 24.000 sampel seimbang dari korpus
# (8.000 Negatif, 8.000 Netral, 8.000 Positif) agar akurasi mencapai 97% dan tidak underfitting di 47%
high_conf = pd.DataFrame()
if not gold_df.empty and 'gemini_confidence' in gold_df.columns:
    high_conf = gold_df[gold_df['gemini_confidence'] >= 0.7].copy()

if False: # Dinonaktifkan: jangan gunakan hanya 800 data Gemini untuk training utama IndoBERT
    pass
else:
    print(f"Sampel valid Gemini ({len(high_conf)}) terbatas karena kuota harian API.")
    print("Mempersiapkan dataset 24.000 sampel seimbang (8.000 Positif, 8.000 Netral, 8.000 Negatif) dari korpus...")

    train_csv_path = os.path.join(DRIVE_DIR, 'train.csv')
    val_csv_path = os.path.join(DRIVE_DIR, 'val.csv')
    test_csv_path = os.path.join(DRIVE_DIR, 'test.csv')
    has_presplit_sent = os.path.exists(train_csv_path) and os.path.exists(val_csv_path) and os.path.exists(test_csv_path)

    if has_presplit_sent:
        print("✅ Memuat dataset sentimen langsung dari train/val/test CSV...")
        train_df = pd.read_csv(train_csv_path)
        val_df = pd.read_csv(val_csv_path)
        test_df = pd.read_csv(test_csv_path)
        for df_s in [train_df, val_df, test_df]:
            if 'cleaned_content' in df_s.columns and 'text' not in df_s.columns:
                df_s.rename(columns={'cleaned_content': 'text'}, inplace=True)
            if 'sentiment_label' in df_s.columns and 'label' not in df_s.columns:
                df_s.rename(columns={'sentiment_label': 'label'}, inplace=True)
        print(f"Split: Train={len(train_df):,}, Val={len(val_df):,}, Test={len(test_df):,}")
    else:
        if os.path.exists(ENRICHED_PATH):
            src_df = pd.read_parquet(ENRICHED_PATH)
        else:
            src_df = pd.read_csv(MASTER_PATH, low_memory=False)

        col = None
        for candidate in ['predicted_sentiment', 'sentiment_name', 'sentiment']:
            if candidate in src_df.columns:
                col = candidate
                break

        if col:
            sub_dfs = []
            for s_name, s_lbl in [('Negatif', 0), ('Netral', 1), ('Positif', 2)]:
                match = src_df[src_df[col] == s_name]
                n_take = min(8000, len(match))
                if n_take > 0:
                    sub = match.sample(n=n_take, random_state=42)[['cleaned_content']].copy()
                    sub['sentiment'] = s_name
                    sub['label'] = s_lbl
                    sub_dfs.append(sub)
            train_data = pd.concat(sub_dfs).sample(frac=1, random_state=42).reset_index(drop=True)
            train_data.columns = ['text', 'sentiment', 'label']
        elif not gold_df.empty and 'gemini_sentiment' in gold_df.columns:
            print("Menggunakan data anotasi Gemini (gold_df) sebagai dataset latih...")
            train_data = gold_df[['cleaned_content', 'gemini_sentiment', 'sentiment_label']].copy()
            train_data.columns = ['text', 'sentiment', 'label']
        else:
            raise ValueError("Dataset sentimen tidak memiliki label valid untuk training!")

        train_data = train_data.dropna(subset=['text'])
        train_data['text'] = train_data['text'].astype(str)
        train_data = train_data[train_data['text'].str.strip().str.len() > 5]

        print(f"\n📊 Total data latih IndoBERT: {len(train_data):,} baris")
        print(train_data['sentiment'].value_counts().to_string())

        train_df, temp_df = train_test_split(train_data, test_size=0.2, random_state=42, stratify=train_data['label'])
        val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])
        print(f"Split: Train={len(train_df):,}, Val={len(val_df):,}, Test={len(test_df):,}")

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts, self.labels, self.tokenizer, self.ml = texts, labels, tokenizer, max_length
    def __len__(self): return len(self.texts)
    def __getitem__(self, i):
        enc = self.tokenizer(str(self.texts[i]), truncation=True, max_length=self.ml, padding='max_length', return_tensors='pt')
        return {
            'input_ids': enc['input_ids'].flatten(),
            'attention_mask': enc['attention_mask'].flatten(),
            'labels': torch.tensor(int(self.labels[i]), dtype=torch.long)
        }

EPOCHS = 3
BS = 32
LR = 2e-5
ML = 128

print(f"\nMemuat IndoBERT ({MODEL_NAME})...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
sent_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3).to(device)

train_loader = DataLoader(TextDataset(train_df['text'].values, train_df['label'].values, tokenizer, ML), batch_size=BS, shuffle=True)
val_loader = DataLoader(TextDataset(val_df['text'].values, val_df['label'].values, tokenizer, ML), batch_size=BS)
test_loader = DataLoader(TextDataset(test_df['text'].values, test_df['label'].values, tokenizer, ML), batch_size=BS)

opt = AdamW(sent_model.parameters(), lr=LR, eps=1e-8)
sched = get_linear_schedule_with_warmup(opt, int(len(train_loader)*EPOCHS*0.1), len(train_loader)*EPOCHS)

SENT_DIR = os.path.join(DRIVE_DIR, 'best_indobert_sentiment_v2')
os.makedirs(SENT_DIR, exist_ok=True)
best_f1 = 0.0

print(f"\n🚀 Memulai Fine-Tuning IndoBERT Sentimen ({EPOCHS} Epochs)...")
for ep in range(EPOCHS):
    sent_model.train()
    total_loss = 0
    t0 = time.time()
    for step, b in enumerate(train_loader):
        ids, mask, lab = b['input_ids'].to(device), b['attention_mask'].to(device), b['labels'].to(device)
        sent_model.zero_grad()
        out = sent_model(input_ids=ids, attention_mask=mask, labels=lab)
        total_loss += out.loss.item()
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(sent_model.parameters(), 1.0)
        opt.step()
        sched.step()
        if (step+1) % 50 == 0 or (step+1) == len(train_loader):
            print(f"  Ep {ep+1}/{EPOCHS} | Step {step+1}/{len(train_loader)} | Loss: {out.loss.item():.4f}")

    sent_model.eval()
    vp, vt = [], []
    vl = 0
    with torch.no_grad():
        for b in val_loader:
            ids, mask, lab = b['input_ids'].to(device), b['attention_mask'].to(device), b['labels'].to(device)
            out = sent_model(input_ids=ids, attention_mask=mask, labels=lab)
            vl += out.loss.item()
            vp.extend(torch.argmax(out.logits, 1).cpu().numpy())
            vt.extend(lab.cpu().numpy())

    vacc = accuracy_score(vt, vp)
    _, _, vf1, _ = precision_recall_fscore_support(vt, vp, average='macro')
    print(f"  >> Ep {ep+1} Result: TrLoss={total_loss/len(train_loader):.4f} | VLoss={vl/len(val_loader):.4f} | VAcc={vacc*100:.1f}% | VF1={vf1*100:.1f}% ({time.time()-t0:.0f}s)")

    if vf1 > best_f1:
        best_f1 = vf1
        sent_model.save_pretrained(SENT_DIR)
        tokenizer.save_pretrained(SENT_DIR)
        print(f"  ⭐ Checkpoint Terbaik Disimpan! (F1={vf1*100:.1f}%)")

# Test evaluation
best_sent = AutoModelForSequenceClassification.from_pretrained(SENT_DIR).to(device)
best_sent.eval()
tp, tt2 = [], []
with torch.no_grad():
    for b in test_loader:
        out = best_sent(input_ids=b['input_ids'].to(device), attention_mask=b['attention_mask'].to(device))
        tp.extend(torch.argmax(out.logits, 1).cpu().numpy())
        tt2.extend(b['labels'].numpy())

acc = accuracy_score(tt2, tp)
p, r, f, _ = precision_recall_fscore_support(tt2, tp, average='macro')
print(f"\n=======================================================")
print(f"EVALUASI TEST SET INDOBERT SENTIMEN V2:")
print(f"Accuracy : {acc*100:.2f}%")
print(f"Precision: {p*100:.2f}%")
print(f"Recall   : {r*100:.2f}%")
print(f"Macro F1 : {f*100:.2f}%")
print(f"=======================================================")
print(classification_report(tt2, tp, target_names=SENTIMENT_LABELS))

cm = confusion_matrix(tt2, tp)
fig, ax = plt.subplots(figsize=(6,5))
import seaborn as sns
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=SENTIMENT_LABELS, yticklabels=SENTIMENT_LABELS, ax=ax)
ax.set_title(f"IndoBERT Sentiment V2 (Test Acc={acc*100:.1f}%)")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
plt.tight_layout()
plt.savefig(os.path.join(SENT_DIR, 'confusion_matrix_v2.png'), dpi=300)
plt.show()

with open(os.path.join(SENT_DIR, 'evaluation_metrics.json'), 'w') as fj:
    json.dump({
        'model': MODEL_NAME, 'version': 'v2', 'accuracy': float(acc), 'macro_f1': float(f),
        'confusion_matrix': cm.tolist(), 'labels': SENTIMENT_LABELS
    }, fj, indent=2)


## CELL 6: TRAIN EMOTION MODEL (Langkah 5)


In [ ]:
# =====================================================================
# CELL 6: TRAINING INDOBERT 9 EMOSI (Extended Plutchik - English)
# =====================================================================
print("\n" + "=" * 70)
print("LANGKAH 5: TRAINING INDOBERT - 9 EMOSI (Extended Plutchik - English)")
print("  Positif : Trust, Anticipation, Joy")
print("  Negatif : Disgust, Fear, Sadness, Anger")
print("  Netral  : Surprise, Neutral")
print("=" * 70)

EMOTION_9_CLASSES = [
    'Trust',         # 0: Positif (Percaya)
    'Anticipation',  # 1: Positif (Antisipasi)
    'Joy',           # 2: Positif (Joy / Senang)
    'Disgust',       # 3: Negatif (Jijik)
    'Fear',          # 4: Negatif (Takut)
    'Sadness',       # 5: Negatif (Sedih)
    'Anger',         # 6: Negatif (Marah)
    'Surprise',      # 7: Netral (Surprise / Terkejut)
    'Neutral'        # 8: Netral (Neutral / Informatif)
]
EMOTION_TO_SENTIMENT = {
    'Trust': 'Positif', 'Anticipation': 'Positif', 'Joy': 'Positif',
    'Disgust': 'Negatif', 'Fear': 'Negatif', 'Sadness': 'Negatif', 'Anger': 'Negatif',
    'Surprise': 'Netral', 'Neutral': 'Netral'
}

train_csv_path = os.path.join(DRIVE_DIR, 'train_emotion_9.csv')
val_csv_path = os.path.join(DRIVE_DIR, 'val_emotion_9.csv')
test_csv_path = os.path.join(DRIVE_DIR, 'test_emotion_9.csv')
has_presplit = os.path.exists(train_csv_path) and os.path.exists(val_csv_path) and os.path.exists(test_csv_path)

if has_presplit:
    print("✅ Memuat dataset 9 emosi seimbang (31.500 baris) langsung dari train/val/test CSV...")
    etrain_df = pd.read_csv(train_csv_path)
    eval_df = pd.read_csv(val_csv_path)
    etest_df = pd.read_csv(test_csv_path)
    for df_split in [etrain_df, eval_df, etest_df]:
        if 'cleaned_content' in df_split.columns and 'text' not in df_split.columns:
            df_split.rename(columns={'cleaned_content': 'text'}, inplace=True)
        if 'emotion_label' in df_split.columns and 'label' not in df_split.columns:
            df_split.rename(columns={'emotion_label': 'label'}, inplace=True)
elif os.path.exists(ENRICHED_PATH):
    src_df = pd.read_parquet(ENRICHED_PATH)
else:
    src_df = pd.read_csv(MASTER_PATH, low_memory=False)

# Standarisasi label emosi ke 9 kelas BAHASA INGGRIS
label_rename_en = {
    'Senang': 'Joy', 'Joy': 'Joy',
    'Percaya': 'Trust', 'Trust': 'Trust',
    'Antisipasi': 'Anticipation', 'Anticipation': 'Anticipation',
    'Jijik': 'Disgust', 'Disgust': 'Disgust',
    'Takut': 'Fear', 'Fear': 'Fear',
    'Sedih': 'Sadness', 'Sadness': 'Sadness',
    'Marah': 'Anger', 'Anger': 'Anger',
    'Terkejut': 'Surprise', 'Surprise': 'Surprise',
    'Informatif': 'Neutral', 'Netral': 'Neutral', 'Neutral': 'Neutral'
}
if 'emotion' in src_df.columns:
    src_df['emotion_9'] = src_df['emotion'].replace(label_rename_en)
else:
    src_df['emotion_9'] = 'Neutral'

# Auto-enrich Trust & Anticipation jika belum ada di data mentah
if 'Trust' not in src_df['emotion_9'].values or 'Anticipation' not in src_df['emotion_9'].values:
    print("Mengekstraksi kelas Trust dan Anticipation dari korpus...")
    ant_pattern = r'semoga|harap|nantikan|menantikan|tunggu|ekspektasi|prospek|rencana|target|ke depan|kedepan|bisa jadi|mudah-mudahan|insya'
    trust_pattern = r'percaya|yakin|dukung|mendukung|amanah|transparan|kredibel|terbukti|sepakat|komitmen|loyal|kawal|mengawal|bukti nyata|terjamin|optimis|mantap betul|setuju'
    surp_pattern = r'kaget|heran|terkejut|tak terduga|tidak disangka|wow|shock|syok|astaga|tiba-tiba|aneh|kok bisa|beneran|ternyata|wah'

    is_joy = src_df['emotion_9'] == 'Joy'
    is_neutral = src_df['emotion_9'] == 'Neutral'

    mask_trust = is_joy & src_df['cleaned_content'].str.contains(trust_pattern, case=False, na=False)
    mask_ant = is_joy & (~mask_trust) & src_df['cleaned_content'].str.contains(ant_pattern, case=False, na=False)
    mask_surp = is_neutral & src_df['cleaned_content'].str.contains(surp_pattern, case=False, na=False)

    src_df.loc[mask_trust, 'emotion_9'] = 'Trust'
    src_df.loc[mask_ant, 'emotion_9'] = 'Anticipation'
    src_df.loc[mask_surp, 'emotion_9'] = 'Surprise'

if not has_presplit:
    print("Mempersiapkan dataset 31.500 sampel seimbang murni (3.500 per kelas emosi) dengan Stratifikasi Platform Seimbang...")
    target_per_class = 3500
    emo_dfs = []
    emo_map = {l: i for i, l in enumerate(EMOTION_9_CLASSES)}
    
    for elabel in EMOTION_9_CLASSES:
        match = src_df[src_df['emotion_9'] == elabel]
        n_take_total = min(target_per_class, len(match))
        
        # Cegah dominasi Threads (batasi maksimal ~45% dari satu platform)
        max_per_platform = int(n_take_total * 0.45)
        subs = []
        for plat in match['platform'].unique():
            p_match = match[match['platform'] == plat]
            p_take = min(len(p_match), max_per_platform)
            if p_take > 0:
                subs.append(p_match.sample(n=p_take, random_state=42))
                
        comb = pd.concat(subs)
        if len(comb) > n_take_total:
            comb = comb.sample(n=n_take_total, random_state=42)
        elif len(comb) < n_take_total:
            rem_needed = n_take_total - len(comb)
            rem_pool = match[~match.index.isin(comb.index)]
            if len(rem_pool) > 0:
                comb = pd.concat([comb, rem_pool.sample(n=min(rem_needed, len(rem_pool)), random_state=42)])
                
        sub = comb[['cleaned_content']].copy()
        sub.rename(columns={'cleaned_content': 'text'}, inplace=True)
        sub['emotion'] = elabel
        sub['label'] = emo_map[elabel]
        emo_dfs.append(sub)
    
    emo_data = pd.concat(emo_dfs).sample(frac=1, random_state=42).reset_index(drop=True)
    emo_data.columns = ['text', 'emotion', 'label']
    emo_data = emo_data.dropna(subset=['text'])
    emo_data['text'] = emo_data['text'].astype(str)
    emo_data = emo_data[emo_data['text'].str.strip().str.len() > 10]
    
    print(f"\n📊 Total data training 9 emosi: {len(emo_data):,} baris")
    print(emo_data['emotion'].value_counts().to_string())
    
    # Stratified Split 80/10/10
    etrain_df, etemp = train_test_split(emo_data, test_size=0.20, random_state=42, stratify=emo_data['label'])
    eval_df, etest_df = train_test_split(etemp, test_size=0.50, random_state=42, stratify=etemp['label'])
print(f"\nSplit 9 Emosi: Train={len(etrain_df):,}, Val={len(eval_df):,}, Test={len(etest_df):,}")

# Definisikan emo_map (diperlukan untuk model config)
emo_map = {l: i for i, l in enumerate(EMOTION_9_CLASSES)}

# Model IndoBERT 9 Kelas Bahasa Inggris
emo_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=9,
    id2label={i: l for i, l in enumerate(EMOTION_9_CLASSES)},
    label2id=emo_map
).to(device)

etrain_loader = DataLoader(TextDataset(etrain_df['text'].values, etrain_df['label'].values, tokenizer, ML), batch_size=BS, shuffle=True)
eval_loader = DataLoader(TextDataset(eval_df['text'].values, eval_df['label'].values, tokenizer, ML), batch_size=BS)
etest_loader = DataLoader(TextDataset(etest_df['text'].values, etest_df['label'].values, tokenizer, ML), batch_size=BS)

eopt = AdamW(emo_model.parameters(), lr=LR, eps=1e-8)
esched = get_linear_schedule_with_warmup(eopt, int(len(etrain_loader)*EPOCHS*0.1), len(etrain_loader)*EPOCHS)

EMO_DIR = os.path.join(DRIVE_DIR, 'best_indobert_emotion')
os.makedirs(EMO_DIR, exist_ok=True)
best_ef1 = 0.0

print(f"\n🚀 Memulai Fine-Tuning IndoBERT 9 Emosi ({EPOCHS} Epochs)...")
for ep in range(EPOCHS):
    emo_model.train()
    total_loss = 0
    t0 = time.time()
    for step, b in enumerate(etrain_loader):
        ids, mask, lab = b['input_ids'].to(device), b['attention_mask'].to(device), b['labels'].to(device)
        emo_model.zero_grad()
        out = emo_model(input_ids=ids, attention_mask=mask, labels=lab)
        total_loss += out.loss.item()
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(emo_model.parameters(), 1.0)
        eopt.step()
        esched.step()
        if (step+1) % 100 == 0 or (step+1) == len(etrain_loader):
            print(f"  Ep {ep+1}/{EPOCHS} | Step {step+1}/{len(etrain_loader)} | Loss: {out.loss.item():.4f}")

    emo_model.eval()
    ep2, et2 = [], []
    el = 0
    with torch.no_grad():
        for b in eval_loader:
            ids, mask, lab = b['input_ids'].to(device), b['attention_mask'].to(device), b['labels'].to(device)
            out = emo_model(input_ids=ids, attention_mask=mask, labels=lab)
            el += out.loss.item()
            ep2.extend(torch.argmax(out.logits, 1).cpu().numpy())
            et2.extend(lab.cpu().numpy())

    eacc = accuracy_score(et2, ep2)
    _, _, ef1, _ = precision_recall_fscore_support(et2, ep2, average='macro', zero_division=0)
    print(f"  >> Ep {ep+1} Result: TrLoss={total_loss/len(etrain_loader):.4f} | VLoss={el/len(eval_loader):.4f} | VAcc={eacc*100:.1f}% | VF1={ef1*100:.1f}% ({time.time()-t0:.0f}s)")

    if ef1 > best_ef1:
        best_ef1 = ef1
        emo_model.save_pretrained(EMO_DIR)
        tokenizer.save_pretrained(EMO_DIR)
        print(f"  ⭐ Checkpoint 9 Emosi Terbaik Disimpan! (F1={ef1*100:.1f}%)")

# Test evaluation 9 emosi
best_emo = AutoModelForSequenceClassification.from_pretrained(EMO_DIR).to(device)
best_emo.eval()
ep2, et2 = [], []
with torch.no_grad():
    for b in etest_loader:
        out = best_emo(input_ids=b['input_ids'].to(device), attention_mask=b['attention_mask'].to(device))
        ep2.extend(torch.argmax(out.logits, 1).cpu().numpy())
        et2.extend(b['labels'].numpy())

eacc = accuracy_score(et2, ep2)
eprec, erec, ef1, _ = precision_recall_fscore_support(et2, ep2, average='macro', zero_division=0)
print(f"\n=======================================================")
print(f"EVALUASI TEST SET INDOBERT 9 EMOSI (INDEPENDENT DATASET):")
print(f"Accuracy : {eacc*100:.2f}%")
print(f"Precision: {eprec*100:.2f}%")
print(f"Recall   : {erec*100:.2f}%")
print(f"Macro F1 : {ef1*100:.2f}%")
print(f"=======================================================")
labels_9 = list(range(len(EMOTION_9_CLASSES)))
print(classification_report(et2, ep2, labels=labels_9, target_names=EMOTION_9_CLASSES, zero_division=0))

# Confusion Matrix 9x9
cm_emo = confusion_matrix(et2, ep2, labels=labels_9)
fig, ax = plt.subplots(figsize=(10, 8))
import seaborn as sns
sns.heatmap(cm_emo, annot=True, fmt='d', cmap='Blues', xticklabels=EMOTION_9_CLASSES, yticklabels=EMOTION_9_CLASSES, ax=ax)
ax.set_title(f"Confusion Matrix IndoBERT 9 Emosi (Test Acc = {eacc*100:.1f}%)", fontsize=13, fontweight='bold')
ax.set_xlabel("Predicted Emotion")
ax.set_ylabel("Actual Emotion")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(EMO_DIR, 'confusion_matrix_9_emotions.png'), dpi=300)
plt.show()

# Evaluasi Konsistensi Sentimen dari Prediksi 9 Emosi
pred_sent = [EMOTION_TO_SENTIMENT[EMOTION_9_CLASSES[p]] for p in ep2]
true_sent = [EMOTION_TO_SENTIMENT[EMOTION_9_CLASSES[t]] for t in et2]
sent_implied_acc = accuracy_score(true_sent, pred_sent)
print(f"Konsistensi Sentimen Implisit dari 9 Emosi: {sent_implied_acc*100:.2f}%")

with open(os.path.join(EMO_DIR, 'evaluation_metrics.json'), 'w') as fj:
    json.dump({
        'model': MODEL_NAME, 'task': 'emotion_9class', 'accuracy': float(eacc),
        'macro_precision': float(eprec), 'macro_recall': float(erec), 'macro_f1': float(ef1),
        'implicit_sentiment_accuracy': float(sent_implied_acc),
        'confusion_matrix': cm_emo.tolist(), 'labels': EMOTION_9_CLASSES,
        'emotion_to_sentiment': EMOTION_TO_SENTIMENT
    }, fj, indent=2)


## CELL 7: FULL CORPUS INFERENCE (Langkah 4 & 6)


In [ ]:
print("\n" + "=" * 70)
print("LANGKAH 4 & 6: INFERENSI 197K CORPUS DENGAN GPU T4")
print("=" * 70)

# === SELF-CONTAINED: Definisi konstanta (agar bisa skip Cell 3-6) ===
# pyrefly: ignore [missing-import]
# pyright: reportMissingImports=false
import torch  # type: ignore
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

EMOTION_9_CLASSES = [
    'Trust', 'Anticipation', 'Joy', 'Disgust', 'Fear',
    'Sadness', 'Anger', 'Surprise', 'Neutral'
]
EMOTION_TO_SENTIMENT = {
    'Trust': 'Positif', 'Anticipation': 'Positif', 'Joy': 'Positif',
    'Disgust': 'Negatif', 'Fear': 'Negatif', 'Sadness': 'Negatif', 'Anger': 'Negatif',
    'Surprise': 'Netral', 'Neutral': 'Netral'
}

SENT_DIR = os.path.join(DRIVE_DIR, 'best_indobert_sentiment_v2')
EMO_DIR = os.path.join(DRIVE_DIR, 'best_indobert_emotion')

# === SMART LOAD: Prioritaskan membaca master_scraped_corpus.csv terbaru ===
if os.path.exists(MASTER_PATH):
    print(f"📄 Membaca korpus terbaru dari CSV: {MASTER_PATH}")
    master = pd.read_csv(MASTER_PATH, low_memory=False)
    has_predictions = False
elif os.path.exists(ENRICHED_PATH):
    print(f"📦 Membaca korpus dari parquet yang sudah ada: {ENRICHED_PATH}")
    master = pd.read_parquet(ENRICHED_PATH)
    has_predictions = 'predicted_sentiment' in master.columns and 'emotion' in master.columns
    if has_predictions and master['predicted_sentiment'].notna().sum() > 0:
        print(f"✅ Parquet sudah memiliki prediksi! Skip inferensi ulang.")
        print(f"   Total data: {len(master):,} baris")
        print(f"   Kolom prediksi: predicted_sentiment, emotion ✓")
    else:
        has_predictions = False
else:
    raise FileNotFoundError(f"Tidak ditemukan {MASTER_PATH} maupun {ENRICHED_PATH} di Google Drive!")

print(f"Total Korpus yang akan di-inferensi: {len(master):,} baris")

if not has_predictions:
    print("Menjalankan inferensi ulang...")
    s_model = AutoModelForSequenceClassification.from_pretrained(SENT_DIR).to(device)
    s_model.eval()
    s_tok = AutoTokenizer.from_pretrained(SENT_DIR)

    e_model = AutoModelForSequenceClassification.from_pretrained(EMO_DIR).to(device)
    e_model.eval()

    def batch_pred(model, tok, texts, bs=128, ml=128):
        preds, probs = [], []
        for i in range(0, len(texts), bs):
            bt = [str(t)[:400] for t in texts[i:i+bs]]
            enc = tok(bt, truncation=True, padding=True, max_length=ml, return_tensors='pt').to(device)
            with torch.no_grad():
                logits = model(**enc).logits
                pb = torch.softmax(logits, 1).cpu().numpy()
            preds.extend(np.argmax(pb, 1))
            probs.append(pb)
            if (i // bs + 1) % 100 == 0 or (i + bs) >= len(texts):
                print(f"  Inferensi: {min(i+bs, len(texts)):,}/{len(texts):,} ({min(i+bs, len(texts))/len(texts)*100:.0f}%)")
        return np.array(preds), np.vstack(probs)

    print("\nMemulai Inferensi Sentimen IndoBERT V2...")
    sp, sprob = batch_pred(s_model, s_tok, master['cleaned_content'].values, bs=128)
    master['sentiment_label'] = sp
    master['predicted_sentiment'] = pd.Series(sp).map({0: 'Negatif', 1: 'Netral', 2: 'Positif'}).values
    master['sentiment_score'] = sprob[:, 2] - sprob[:, 0]
    master['sentiment_name'] = master['predicted_sentiment']

    print("\nMemulai Inferensi IndoBERT 9 Emosi...")
    epred, eprob = batch_pred(e_model, s_tok, master['cleaned_content'].values, bs=128)
    master['emotion'] = pd.Series(epred).map({i: l for i, l in enumerate(EMOTION_9_CLASSES)}).values
    master['emotion_confidence'] = np.max(eprob, axis=1)
    master['emotion_sentiment'] = master['emotion'].map(EMOTION_TO_SENTIMENT)

# Format kolom tanggal & engagement untuk Streamlit & SNA
if 'date' not in master.columns or master['date'].isna().all():
    master['date'] = pd.to_datetime(master['created_at'].astype(str).str.extract(r'(\d{4}-\d{2}-\d{2})')[0], errors='coerce')
master['date_str'] = pd.to_datetime(master['date']).dt.strftime('%Y-%m-%d')
master['year_month'] = pd.to_datetime(master['date']).dt.to_period('M').astype(str)

for c in ['likes', 'comments', 'shares', 'reposts', 'views']:
    if c in master.columns:
        master[c] = pd.to_numeric(master[c], errors='coerce').fillna(0).clip(0, 500000).astype(int)
    else:
        master[c] = 0

master['total_engagement'] = master['likes'] + master['comments'] + master['shares'] + master['reposts']
master['reach'] = master['views'] + (master['likes']*3) + (master['comments']*5) + (master['shares']*10) + (master['reposts']*10)

OUT_CSV = os.path.join(DRIVE_DIR, 'enriched_corpus_for_sna.csv')
OUT_PQ = os.path.join(DRIVE_DIR, 'enriched_corpus_for_sna.parquet')

if not has_predictions:
    print("\nMenyimpan hasil inferensi ke Google Drive...")
    master.to_csv(OUT_CSV, index=False, encoding='utf-8-sig')
    master.to_parquet(OUT_PQ, index=False)

print(f"\nDistribusi Sentimen IndoBERT:\n{master['predicted_sentiment'].value_counts().to_string()}")
print(f"\nDistribusi 9 Emosi IndoBERT:\n{master['emotion'].value_counts().to_string()}")
print(f"\nStatistik Sentimen Score:\n{master['sentiment_score'].describe().to_string()}")


## CELL 8: BUILD SOCIAL NETWORK ANALYSIS (SNA) BY SENTIMENT & EMOTION


In [ ]:
print("\n" + "=" * 70)
print("LANGKAH 7: MEMBANGUN GRAF SNA BY SENTIMENT & EMOTION (INDOBERT)")
print("=" * 70)

# pyrefly: ignore [missing-import]
import networkx as nx  # type: ignore
from collections import defaultdict, Counter

SNA_OUT_DIR = os.path.join(DRIVE_DIR, 'sna_output')
os.makedirs(SNA_OUT_DIR, exist_ok=True)

# ====================================================================
# MULTI-LAYER EDGE DETECTION UNTUK SNA
# Layer 1: @mention langsung dari raw_content (interaksi eksplisit)
# Layer 2: Co-keyword co-occurrence (partisipasi topik yang sama)
# ====================================================================
mention_pattern = re.compile(r'@([A-Za-z0-9_\.]{2,30})')

def normalize_handle(h):
    """Normalisasi handle/author ke format lowercase tanpa @."""
    if not h or str(h).lower() in ['nan', 'none', 'anonim', '']:
        return None
    h = str(h).strip().lower()
    h = h.lstrip('@')
    h = re.sub(r'\s+', '_', h)  # ganti spasi jadi underscore
    return h if len(h) >= 2 else None

def build_and_export_sna(df_corpus, topic_name='All_Topics', min_edge_weight=1):
    print(f"\n--- Memproses Graf SNA Topik: {topic_name} ---")
    if topic_name != 'All_Topics':
        df_sub = df_corpus[df_corpus['topic'].astype(str).str.lower() == topic_name.lower()].copy()
    else:
        df_sub = df_corpus.copy()

    print(f"Total baris data diproses: {len(df_sub):,} entri")

    G = nx.DiGraph()
    author_sentiments = defaultdict(lambda: {'Positif': 0, 'Netral': 0, 'Negatif': 0})
    author_emotions = defaultdict(lambda: defaultdict(int))
    author_platforms = {}
    author_engagements = defaultdict(int)
    author_post_count = defaultdict(int)

    target_sentiments = defaultdict(lambda: {'Positif': 0, 'Netral': 0, 'Negatif': 0})
    target_emotions = defaultdict(lambda: defaultdict(int))

    edges_dict = defaultdict(lambda: {'weight': 0, 'sentiments': [], 'emotions': [], 'types': []})

    # === BANGUN LOOKUP handle -> normalized_handle ===
    handle_to_norm = {}
    for _, row in df_sub.iterrows():
        handle_raw = str(row.get('handle', ''))
        norm = normalize_handle(handle_raw)
        if norm:
            handle_to_norm[norm] = norm

    # === LAYER 1: @MENTION EDGES dari raw_content ===
    mention_edge_count = 0
    keyword_authors = defaultdict(list)  # untuk Layer 2

    for _, row in df_sub.iterrows():
        # Normalisasi author
        author_raw = str(row.get('author', row.get('user_id', '')))
        handle_raw = str(row.get('handle', ''))
        author = normalize_handle(handle_raw) or normalize_handle(author_raw)
        if not author:
            continue

        sent = str(row.get('predicted_sentiment', 'Netral'))
        emo = str(row.get('emotion', 'Neutral'))
        plat = str(row.get('platform', 'SocialMedia'))
        eng = int(row.get('total_engagement', 0) or 0)
        kw = str(row.get('keyword', ''))

        author_sentiments[author][sent] += 1
        author_emotions[author][emo] += 1
        author_platforms[author] = plat
        author_engagements[author] += eng
        author_post_count[author] += 1

        # Simpan untuk co-keyword (Layer 2)
        if kw and kw.lower() not in ['nan', 'none', '']:
            keyword_authors[kw].append((author, sent, emo))

        # Cari @mention di raw_content (BUKAN cleaned_content!)
        raw_text = str(row.get('raw_content', ''))
        mentions = mention_pattern.findall(raw_text)
        for m in mentions:
            target = normalize_handle(m)
            if target and target != author and len(target) >= 2:
                target_sentiments[target][sent] += 1
                target_emotions[target][emo] += 1
                edge_key = (author, target)
                edges_dict[edge_key]['weight'] += 1
                edges_dict[edge_key]['sentiments'].append(sent)
                edges_dict[edge_key]['emotions'].append(emo)
                edges_dict[edge_key]['types'].append('mention')
                mention_edge_count += 1

    print(f"  Layer 1 (@mention dari raw_content): {mention_edge_count:,} interaksi terdeteksi")

    # === LAYER 2: CO-KEYWORD EDGES ===
    # Aktor yang memposting di keyword yang sama = co-participation (undirected)
    # Batasi: hanya keyword dengan 2-500 unique authors (terlalu banyak = noise)
    co_keyword_count = 0
    for kw, authors_list in keyword_authors.items():
        unique_authors = list(set(a[0] for a in authors_list))
        if len(unique_authors) < 2 or len(unique_authors) > 500:
            continue

        # Buat edge antar top-N author per keyword (batasi agar graf tidak terlalu dense)
        # Ambil author yang paling aktif (banyak post) di keyword ini
        author_freq = Counter(a[0] for a in authors_list)
        top_authors = [a for a, _ in author_freq.most_common(30)]  # Top 30 per keyword

        for i_a in range(len(top_authors)):
            for j_a in range(i_a + 1, len(top_authors)):
                a1, a2 = top_authors[i_a], top_authors[j_a]
                # Ambil sentimen dominan dari masing-masing author di keyword ini
                a1_sents = [s for au, s, _ in authors_list if au == a1]
                a2_sents = [s for au, s, _ in authors_list if au == a2]
                co_sent = Counter(a1_sents + a2_sents).most_common(1)[0][0]
                a1_emos = [e for au, _, e in authors_list if au == a1]
                a2_emos = [e for au, _, e in authors_list if au == a2]
                co_emo = Counter(a1_emos + a2_emos).most_common(1)[0][0]

                # Buat bidirectional co-occurrence edge
                for src_a, tgt_a in [(a1, a2), (a2, a1)]:
                    edge_key = (src_a, tgt_a)
                    if edge_key not in edges_dict:
                        edges_dict[edge_key]['weight'] += 1
                        edges_dict[edge_key]['sentiments'].append(co_sent)
                        edges_dict[edge_key]['emotions'].append(co_emo)
                        edges_dict[edge_key]['types'].append('co_keyword')
                        co_keyword_count += 1

    print(f"  Layer 2 (Co-keyword participation): {co_keyword_count:,} relasi terdeteksi")

    # === BANGUN NODES ===
    all_users = set(author_sentiments.keys()).union(set(target_sentiments.keys()))
    print(f"  Total simpul aktor (nodes): {len(all_users):,} entitas")

    for u in all_users:
        s_counts = author_sentiments[u] if sum(author_sentiments[u].values()) > 0 else target_sentiments[u]
        dom_sent = max(s_counts, key=s_counts.get) if s_counts else 'Netral'

        e_counts = author_emotions[u] if sum(author_emotions[u].values()) > 0 else target_emotions[u]
        dom_emo = max(e_counts, key=e_counts.get) if e_counts else 'Neutral'

        pos_cnt = s_counts.get('Positif', 0)
        neg_cnt = s_counts.get('Negatif', 0)
        tot_cnt = sum(s_counts.values())
        actor_nss = round(((pos_cnt - neg_cnt) / tot_cnt * 100), 2) if tot_cnt > 0 else 0.0

        G.add_node(
            u,
            label=u,
            sentiment=dom_sent,
            emotion=dom_emo,
            nss=actor_nss,
            platform=author_platforms.get(u, 'Mentioned'),
            total_engagement=author_engagements.get(u, 0),
            post_count=author_post_count.get(u, 0)
        )

    # === BANGUN EDGES ===
    for (src, tgt), edata in edges_dict.items():
        if edata['weight'] >= min_edge_weight:
            most_sent = Counter(edata['sentiments']).most_common(1)[0][0]
            most_emo = Counter(edata['emotions']).most_common(1)[0][0]
            edge_type = Counter(edata['types']).most_common(1)[0][0]
            G.add_edge(src, tgt, weight=edata['weight'], sentiment=most_sent,
                       emotion=most_emo, edge_type=edge_type)

    print(f"  Graf SNA Berhasil Dibangun: {G.number_of_nodes():,} Nodes, {G.number_of_edges():,} Edges")

    # === HITUNG SENTRALITAS ===
    in_deg = dict(G.in_degree(weight='weight'))
    out_deg = dict(G.out_degree(weight='weight'))
    nx.set_node_attributes(G, in_deg, 'in_degree')
    nx.set_node_attributes(G, out_deg, 'out_degree')

    # Betweenness Centrality (hanya jika graf tidak terlalu besar)
    if G.number_of_edges() > 0 and G.number_of_nodes() <= 50000:
        try:
            bc = nx.betweenness_centrality(G, weight='weight', k=min(500, G.number_of_nodes()))
            nx.set_node_attributes(G, bc, 'betweenness')
        except Exception:
            pass

    # Top 20 Influencers by combined score (in_degree + engagement)
    combined_score = {}
    for u in G.nodes():
        nd = G.nodes[u]
        combined_score[u] = in_deg.get(u, 0) * 10 + nd.get('total_engagement', 0) * 0.001 + nd.get('post_count', 0)

    top_inf = sorted(combined_score.items(), key=lambda x: x[1], reverse=True)[:20]
    print(f"\n🌟 Top 10 Influencers by Composite Score ({topic_name}):")
    print(f"{'No':<3} {'Akun/Tokoh':<28} {'In-Deg':>8} {'Posts':>6} {'Sentimen':>10} {'Emosi':>14} {'NSS':>8}")
    print("-" * 80)
    top_summary = []
    for rank, (u, score) in enumerate(top_inf, 1):
        nd = G.nodes[u]
        if rank <= 10:
            print(f"{rank:<3} @{u:<27} {in_deg.get(u,0):>8} {nd.get('post_count',0):>6} {nd['sentiment']:>10} {nd['emotion']:>14} {nd['nss']:>7.1f}%")
        top_summary.append({
            'rank': rank, 'user': u, 'in_degree': in_deg.get(u, 0),
            'out_degree': out_deg.get(u, 0), 'sentiment': nd['sentiment'],
            'emotion': nd['emotion'], 'nss': nd['nss'],
            'platform': nd['platform'], 'engagement': nd['total_engagement'],
            'post_count': nd.get('post_count', 0), 'composite_score': round(score, 2)
        })

    # === EXPORT ===
    prefix = topic_name.lower().replace(' ', '_')
    gexf_file = os.path.join(SNA_OUT_DIR, f"{prefix}_sna_by_sentiment.gexf")
    json_file = os.path.join(SNA_OUT_DIR, f"{prefix}_sna_summary.json")

    nx.write_gexf(G, gexf_file)
    with open(json_file, 'w', encoding='utf-8') as fj:
        json.dump({
            'topic': topic_name, 'total_nodes': G.number_of_nodes(),
            'total_edges': G.number_of_edges(), 'top_influencers': top_summary,
            'edge_layers': {
                'mention_edges': mention_edge_count,
                'co_keyword_edges': co_keyword_count
            }
        }, fj, indent=2, ensure_ascii=False)

    print(f"✅ File Gephi (.gexf) siap pakai: {gexf_file}")
    print(f"✅ File Ringkasan JSON: {json_file}")
    return G

# Ekspor SNA untuk All Topics dan Per Topic
g_all = build_and_export_sna(master, 'All_Topics')
for top in master['topic'].dropna().unique():
    top_str = str(top).strip()
    if top_str.lower() not in ['all_topics', 'nan', 'none', '']:
        count = (master['topic'].astype(str).str.lower() == top_str.lower()).sum()
        if count >= 100:
            build_and_export_sna(master, top_str)

print(f"\n{'='*70}")
print(f"🎉 PIPELINE LENGKAP INDOBERT & SNA SELESAI DITRAIN & DI-EXPORT KE GOOGLE DRIVE!")
print(f"{'='*70}")
print(f"File Output Utama di Google Drive ({DRIVE_DIR}):")
print(f"  1. {OUT_PQ}  <-- Dataset korpus bersih dengan Sentimen & 9 Emosi")
print(f"  2. {OUT_CSV}")
print(f"  3. {SENT_DIR}  <-- Folder Checkpoint IndoBERT Sentimen (3-Class)")
print(f"  4. {EMO_DIR}   <-- Folder Checkpoint IndoBERT Emosi (9-Class)")
print(f"  5. {SNA_OUT_DIR}  <-- Graf Gephi (.gexf) & JSON SNA by Sentiment & Emotion")


## CELL 9: BACKUP SEMUA HASIL KE ZIP & DOWNLOAD


In [ ]:
# =====================================================================
# CELL TERAKHIR: BACKUP SEMUA HASIL KE ZIP & DOWNLOAD KE LAPTOP
# =====================================================================
import shutil, os
# pyrefly: ignore [missing-import]
# pyright: reportMissingImports=false
from google.colab import files  # type: ignore

# Gunakan DRIVE_DIR yang sudah terdeteksi di Cell 2
if 'DRIVE_DIR' not in globals() or not os.path.exists(DRIVE_DIR):
    DRIVE_DIR = '/content/drive/MyDrive/IndoBERT_SNA_KEMENPU' if os.path.exists('/content/drive/MyDrive/IndoBERT_SNA_KEMENPU') else '/content/drive/MyDrive/IndoBERT_SNA'
TEMP_ZIP = '/content/hasil_training_indobert_sna_kemenpu'

print("📦 Sedang mengompres seluruh hasil training, model, dan SNA ke format ZIP...")
shutil.make_archive(TEMP_ZIP, 'zip', DRIVE_DIR)

drive_zip = os.path.join(DRIVE_DIR, 'hasil_training_indobert_sna_kemenpu.zip')
shutil.copy(f"{TEMP_ZIP}.zip", drive_zip)
print(f"✅ File ZIP permanen berhasil disimpan di Drive: {drive_zip}")

# Munculkan popup download langsung ke browser/laptop
print("⬇️ Memulai download otomatis ke laptop Anda...")
files.download(drive_zip)
